<br>
Authors: Mychailo & Roberto

#  **New York State of Mind**

##  **Introduction**

This notebook demonstrates **data importing and stratified sampling** from the NYC 311 Service Requests dataset for our project in the module **Data Wrangling (DAW)**.  

The primary dataset used:  

- [NYC 311 Service Requests](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9/about_data)  

---

This notebook covers:

1. **Importing** – retrieving and stratified sampling of real-world open data from NYC Open Data API
2. **Data Validation** – verifying the temporal distribution of the sampled data

---

## Workflow

The notebook implements a **stratified random sampling strategy** to create a representative sample from the full NYC 311 dataset while maintaining:
- Temporal distribution across months
- Geographic distribution across boroughs
- Manageable data size for analysis

-------------------------

## LE1 Importing the Data
###  1. Workflow Overview

1. **Generate month range:**  
   For each year between the selected start and end years, we create `(start, end)` date pairs.  
   First we create quarterly ranges and then split them into the months.
   Example: `2024-01-01T00:00:00` → `2024-01-31T23:59:59` (M1 2024)

2. **Fetch Entries per Day:**  
   For the month we pick a amount of days, which days are random but the amount of days of the month are the same. For the amount of entries we want to fetch per day we calculate the amount as follows: 
   $$ 
      \text{per\_day} = \left\lfloor \frac{ \text{target} \times \text{per\_day\_mult} }{ \lvert \text{days} \rvert } \right\rfloor
   $$

3. **Fetch Random Samples:**  
   For each borough, we randomly pull the calculated per day amount of records from the corresponding day using the openly accessible API. 
   - Data is retrieved via the `.csv` endpoint (faster than JSON).  
   - A random `$offset` and a random choice, acsending or descending is used in sampling ensure randomness.

4. **Combine:**  
   The sampled data from all boroughs and days in the months are concatenated into a single combined DataFrame using  
   `pd.concat(all_quarters, ignore_index=True)`.

---

### 2. Key Functions

| Function | Description |
|-----------|--------------|
| `generate_quarters(start_year, end_year)` | Generates quarterly date ranges |
| `month_range(start, end)` | Generates the month range |
| `fetch_month_strat_data(...)` | Fetches a random subset for one borough and day |

---

### Importing

In [ ]:
# Imports for LE1

import sys
import os
sys.path.append('..')

# Change working directory to parent folder for correct file paths
os.chdir('..')


import pandas as pd
from itertools import chain
from daw_nyc.config import Settings, get_settings
from daw_nyc.libs.import_data import  generate_quarters, generate_month_ranges
from daw_nyc.libs.import_data import get_dataset_stratified
from daw_nyc.libs.test_setup import create_test_sample
from daw_nyc.libs.test_setup import plot_distribution
from daw_nyc.libs.test_setup import prepare_date_time

### Constants

In [ ]:
# Constants 
SETTINGS: Settings = get_settings()

print("Loaded config:")
print(f"{SETTINGS.GROUP_BY}")
print(f"{SETTINGS.GROUP_BY_VALUE}")
print(f"{SETTINGS.PLOT_DIST}")

# TODO: Maybe put this into a env variable
# Selection
SELECT_COLUMNS = [
    "unique_key", "created_date", "closed_date", "agency", "agency_name", 
    "complaint_type", "descriptor", "location_type", "incident_zip", 
    "incident_address", "street_name", "cross_street_1", "cross_street_2",
    "intersection_street_1", "intersection_street_2", "address_type", "city", 
    "landmark", "facility_type", "status", "due_date", "resolution_description", 
    "resolution_action_updated_date", "community_board", "bbl", "borough", 
    "x_coordinate_state_plane", "y_coordinate_state_plane", "open_data_channel_type",
    "park_facility_name", "park_borough", "vehicle_type", "taxi_company_borough", 
    "taxi_pick_up_location", "bridge_highway_name", "bridge_highway_direction", 
    "road_ramp", "bridge_highway_segment", "latitude", "longitude", "location"
]

### Fetch and Sample

In [ ]:
# Fetch sample of datasets and parse to Data Frame 

# 1. generate the time ranges:
quarters = generate_quarters(SETTINGS.DEFAULT_SINCE, SETTINGS.DEFAULT_UNTIL)
months = generate_month_ranges(quarters)
# 2. fetch the data 
df_all_calls = get_dataset_stratified(months, SETTINGS, SELECT_COLUMNS)

### Safe data to csv

In [ ]:
df_all_calls.to_csv("data/nyc_311_2024_2025_sample.csv")

### Testing the import of data

To verify that our stratified sampling function worked correctly, we perform the following validation steps:

1. **Load the saved dataset** – Read the CSV file that was created in the previous step
2. **Prepare temporal features** – Parse date columns and create month grouping for analysis
3. **Create test sample** – Generate a smaller test sample in Parquet format for quick validation
4. **Visualize distribution** – Plot the temporal distribution of records by month to verify:
   - Even distribution across months
   - Representative sampling from the original dataset
   - No systematic bias in the sampling process

This validation ensures that our stratified sampling approach maintains the temporal characteristics of the full NYC 311 dataset.

In [ ]:
# Testing the import 
# Read the saved dataset
df_nyc_311_2024_2025 = pd.read_csv("data/nyc_311_2024_2025_sample.csv")

# Prepare date time columns for grouping
df_nyc_311_2024_2025 = prepare_date_time(df_nyc_311_2024_2025)
    
sample = create_test_sample("data/nyc_311_2024_2025_sample.csv", "data/sample-test.parquet")
sample.head()

# Group by month and plot distribution
print(f"Plotting enabled: {SETTINGS.PLOT_DIST}")
for month, group in df_nyc_311_2024_2025.groupby('create_month'):
    print(f"Processing month: {month}, records: {len(group)}")
    # Plot distribution for each month
    plot_distribution(group, month)
